# Actividad 5.  Evaluación de Subagentes Especializados ISO 27001
_____

_Proyecto Integrador MNA 10_

**Equipo 16**:
- Joel
- Roberto
- David

**Fecha**: Junio, 2026

---


## Resumen Ejecutivo

Este notebook evalua tres subagentes especializados en auditoria ISO 27001, comparando su desempeno en 11 modelos de lenguaje disponibles en Amazon Bedrock. El objetivo es identificar el modelo optimo por agente con base en metricas operacionales y de precision.

---

### Subagentes evaluados

Los tres subagentes cubren dominios distintos de la norma ISO 27001:

1. **Agente de Gobernanza**: analiza el cumplimiento de las clausulas 4 al 10 de la norma (contexto organizacional, liderazgo, planificacion, soporte, operacion, evaluacion y mejora).
2. **Agente Organizacional**: evalua los controles del Anexo A orientados a personas y procesos (A.5-A.8, A.15, A.16, A.18), incluyendo politicas, recursos humanos, relacion con proveedores y cumplimiento legal.
3. **Agente Tecnico**: revisa los controles del Anexo A de naturaleza tecnica e infraestructura (A.9-A.14, A.17), que abarcan control de acceso, criptografia, seguridad fisica, operaciones y comunicaciones.

---

### Metodologia de evaluacion

Para cada subagente se ejecutaran las siguientes etapas:

1. **Construccion del prompt**: definicion del rol, criterios de severidad, metodologia de evaluacion y esquema de salida JSON estandarizado.
2. **Generacion del dataset de evaluacion**: 25 casos de prueba distribuidos en cinco niveles de severidad (No Conformidad Mayor, No Conformidad Menor, Conforme, No Evaluable, Oportunidad de Mejora), cada uno con contexto RAG recuperado y salida esperada documentada.
3. **Benchmark por modelo**: cada prompt se ejecuta sobre los 11 modelos de lenguaje configurados, generando una respuesta por caso de prueba. En total se realizan 825 invocaciones por agente (25 casos x 11 modelos) mas 825 llamadas al juez LLM para calcular la precision.
4. **Evaluacion de metricas**: por cada invocacion se registran las siguientes metricas:
   - **Precision**: puntuacion del juez LLM (0-100) que evalua exactitud del ranking, cobertura de evidencia, calidad de recomendaciones y conformidad del formato JSON.
   - **Latencia de invocacion**: tiempo total de respuesta en milisegundos.
   - **TTFT**: tiempo hasta el primer token (Time To First Token), indicador de capacidad de respuesta.
   - **Tokens de entrada y salida**: uso de tokens para estimar costo operacional por modelo.
   - **Longitud de respuesta**: numero de caracteres generados como proxy de verbosidad.

---

### Resultado esperado

Una tabla comparativa por agente que permita seleccionar el modelo con mejor balance entre precision auditora y eficiencia operacional, justificando la eleccion con datos objetivos.

## 1. Configuracion del Entorno

Carga de configuracion desde `config.yaml`, inicializacion del cliente Bedrock y extraccion del catalogo de modelos.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import re
import time
import statistics
from datetime import datetime
from pathlib import Path
import boto3
import numpy as np
import pandas as pd
import random
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from utils import utils as u

In [30]:
# Set the root repository
ROOT = Path.cwd().parents[1]
ISO = ROOT / 'knowledge-base/iso27001/iso-iec-27001-2013-es.yaml'

In [43]:
# Init the client and read the configuration
config = u.load_config('./config.yaml')
bedrock = u.get_bedrock_client(".env")

# Get the models that can handle spanish prompts with reliability
models = {k:v['endpoint'] for k, v in config['models'].items()
          if v['handles_spanish']}

# Validate that the models are OK so we can test them
# validation = u.validate_models(models, bedrock)

## 2. Agente de Gobernanza

### Descripcion del agente

El **Agente de Gobernanza** es el primer subagente especializado del sistema de auditoria ISO 27001. Su funcion es analizar el cumplimiento de las clausulas estructurales de la norma, que definen los requisitos del Sistema de Gestion de Seguridad de la Informacion (SGSI) como un todo:

- **Clausula 4:** Contexto de la organizacion (partes interesadas, alcance, comprension del entorno)
- **Clausula 5:** Liderazgo (compromiso de la alta direccion, politica de seguridad, roles y responsabilidades)
- **Clausula 6:** Planificacion (gestion de riesgos, objetivos de seguridad)
- **Clausula 7:** Soporte (recursos, competencias, comunicacion, documentacion)
- **Clausula 8:** Operacion (ejecucion de controles y tratamiento de riesgos)
- **Clausula 9:** Evaluacion del desempeno (monitoreo, auditorias internas, revision por la direccion)
- **Clausula 10:** Mejora continua (no conformidades y acciones correctivas)

Dado un contexto RAG recuperado y una clausula especifica a evaluar, el agente genera un hallazgo de auditoria estructurado en formato JSON con seis campos: identificador del control, descripcion breve, observaciones, nivel de severidad, recomendaciones y documentos revisados.

---

### Rol del agente de gobernanza en este notebook

Este primer subagente tiene un rol doble en el desarrollo de la actividad:

1. **Evaluacion de gobernanza:** produce hallazgos de auditoria sobre las clausulas 4-10 de ISO 27001 usando los documentos de la organizacion como evidencia.
2. **Construccion del pipeline de evaluacion:** el agente de gobernanza se utiliza como caso de referencia para definir, probar y validar la arquitectura completa del benchmark antes de replicarla en los agentes organizacional y tecnico. Esto incluye la estructura del prompt, el formato del dataset, la logica del juez LLM y el esquema de metricas que se aplicara a los tres agentes.

### 2.1 Prompt del agente

El prompt define al agente como un **Auditor de Certificacion Senior de ISO/IEC 27001:2013** restringido exclusivamente a las clausulas de gestion (4-10). Esta compuesto por cuatro bloques:

1. **Rol y restriccion de alcance:** establece la identidad del agente y le prohibe explicitamente evaluar controles tecnicos del Anexo A.
2. **Criterios de severidad:** define los cinco niveles de calificacion posibles y las condiciones exactas para asignar cada uno.
3. **Metodologia de evaluacion:** instruye al agente a descomponer la clausula en requisitos atomicos, mapear evidencia del contexto recuperado y asignar la calificacion por contraste.
4. **Formato de salida:** exige una respuesta en JSON valido sin texto adicional fuera del bloque.

---

**Entradas esperadas**

El agente recibe dos bloques de texto en el mensaje del usuario:

- `{{RETRIEVED_CONTEXT}}`: arreglo JSON con los fragmentos recuperados por RAG, cada uno con los campos `document` (nombre del archivo fuente) y `content` (texto extraido).
- `{{CLAUSE_TO_EVALUATE}}`: texto de la clausula ISO 27001 que se va a auditar, incluyendo su identificador y descripcion del requisito a evaluar.

**Salida esperada**

El agente devuelve un unico objeto JSON con seis campos:

| Campo | Tipo | Descripcion |
|---|---|---|
| `control_id` | string | Identificador de la clausula evaluada |
| `short_description` | string | Nombre oficial de la clausula |
| `observations` | array de strings | Un hallazgo por elemento, con referencia al documento fuente |
| `ranking` | string | Nivel de severidad asignado (uno de cinco valores fijos) |
| `recommendations` | array de strings | Un paso de remediacion por elemento, comenzando con verbo en imperativo |
| `documents_reviewed` | array de strings | Nombres exactos de los documentos citados en el contexto recuperado |

---

**Ejemplo**

*Entrada:*

```json
// {{RETRIEVED_CONTEXT}}
[
  {
    "document": "Acta_Comite_Seguridad_Q4_2025.pdf",
    "content": "La reunion de la direccion se llevo a cabo en noviembre de 2025. Se revisaron y documentaron las tendencias de incidentes de seguridad y la inteligencia de amenazas externas. El orden del dia de la reunion no incluyo los resultados de las auditorias internas, el estado de las acciones correctivas abiertas ni el desempeno del SGSI frente a los objetivos."
  }
]
```

```
// {{CLAUSE_TO_EVALUATE}}
Clausula 9.3 - Revision por la direccion: Evaluar si la organizacion realiza revisiones por la
direccion que cubran todos los elementos de entrada obligatorios requeridos por ISO 27001:2013.
```

*Salida:*

```json
{
  "control_id": "Clausula 9.3",
  "short_description": "Revision por la direccion",
  "observations": [
    "La organizacion demuestra la ejecucion formal de revisiones por la direccion, segun lo evidenciado en Acta_Comite_Seguridad_Q4_2025.pdf.",
    "El orden del dia omitio los resultados de las auditorias internas, elemento de entrada obligatorio de la Clausula 9.3.",
    "El orden del dia omitio el estado de las acciones correctivas abiertas y el desempeno del SGSI frente a los objetivos, lo que representa un cumplimiento parcial de los requisitos."
  ],
  "ranking": "No conformidad menor",
  "recommendations": [
    "Actualizar la plantilla del orden del dia del comite de seguridad para incluir todos los elementos de entrada obligatorios de la Clausula 9.3.",
    "Asegurar que los resumenes de auditorias internas, los informes de estado de acciones correctivas y las metricas de desempeno del SGSI sean puntos permanentes en la agenda de cada ciclo de revision."
  ],
  "documents_reviewed": [
    "Acta_Comite_Seguridad_Q4_2025.pdf"
  ]
}
```

---

**Origen de cada entrada**

**`{{CLAUSE_TO_EVALUATE}}`** se construye programaticamente desde el archivo `knowledge-base/iso27001/iso-iec-27001-2013-es.yaml`. Este archivo contiene todos los nodos evaluables de la norma en espanol con tres campos utiles por clausula: `description` (el requisito normativo), `components` (los sub-requisitos atomicos en forma de lista a, b, c...) y `ref_id` + `name` (el identificador y nombre oficial). La funcion de construccion combina estos campos en un solo bloque de texto:

> Nota: cuatro clausulas (7.1, 8.2, 8.3, 10.2) no tienen `components` en el YAML y requieren texto manual o quedan excluidas del benchmark.

**`{{RETRIEVED_CONTEXT}}`** proviene del indice vectorial en Amazon S3 Vectors (`iso27001-vectors-e16`, indice `iso27001-compliance-index`). Los documentos de la organizacion (politicas, procedimientos y registros del sponsor) fueron previamente indexados con el modelo de embeddings `amazon.titan-embed-text-v2:0`. En tiempo de ejecucion, el texto de la clausula a evaluar se usa como query para recuperar los `top_k=3` fragmentos mas semanticamente relevantes. Cada fragmento recuperado incluye el nombre del documento fuente y el texto extraido, y se serializa como arreglo JSON antes de inyectarse en el mensaje del usuario.

### 2.2 Metodologia de evaluacion: LLM como juez sin respuesta de referencia

Para medir la precision de los hallazgos generados por el agente se utiliza un segundo modelo LLM actuando como juez. Este juez recibe los mismos insumos que el agente (la clausula y el contexto RAG) junto con la respuesta producida, y emite una calificacion numerica estructurada.

---

#### Enfoque adoptado: evaluacion sin respuesta de referencia

El juez evalua la respuesta del agente unicamente a partir de tres entradas:

- `{{CLAUSE_TO_EVALUATE}}`: la clausula ISO 27001 que se auditaba
- `{{RETRIEVED_CONTEXT}}`: la evidencia documental disponible en el momento de la evaluacion
- `{{AGENT_RESPONSE}}`: el hallazgo JSON producido por el agente

No se utiliza una respuesta correcta predefinida ("gold answer"). El juez razona desde la misma evidencia que el agente y determina si el hallazgo es coherente con ella.

---

#### Por que esta decision

El enfoque alternativo, evaluar contra una respuesta de referencia elaborada por un experto humano, presenta tres problemas estructurales para este proyecto:

1. **Ambiguedad valida en auditoria ISO 27001.** Para una misma clausula y un mismo conjunto de evidencia, un auditor experto podria asignar "No conformidad menor" y otro "Oportunidad de Mejora" y ambos estarian en lo correcto. Un juez basado en referencia penalizaria una respuesta valida por no coincidir con la redaccion exacta del gold answer.

2. **Costo de construccion del dataset.** Crear 25 respuestas de referencia de calidad por agente requiere revision experta para cada caso. Un dataset de referencia mal construido introduce ruido sistematico que invalida el benchmark completo.

3. **Lo que importa medir es el razonamiento, no la similitud.** El objetivo del benchmark es identificar que modelo razona mejor a partir de evidencia documental, no que modelo reproduce mejor una redaccion predeterminada.

El enfoque sin referencia evalua directamente lo que interesa: si el ranking es coherente con la evidencia, si las observaciones estan fundamentadas en los documentos (sin alucinaciones) y si las recomendaciones son accionables.

---

#### Criterios de evaluacion del juez (100 puntos)

| Criterio | Puntos | Pregunta que responde el juez |
|---|---|---|
| Coherencia del ranking | 40 | El nivel de severidad asignado, es logicamente justificado por la evidencia y los requisitos de la clausula? |
| Fundamentacion en evidencia | 30 | Las observaciones estan trazadas a documentos reales del contexto? Hay contenido fabricado? |
| Especificidad de recomendaciones | 20 | Las recomendaciones son accionables y abordan la brecha concreta identificada? |
| Cumplimiento del formato | 10 | Los seis campos JSON estan presentes y completados correctamente? |

El campo `ranking_match` en la salida del juez indica si el ranking fue plenamente coherente (puntuacion completa en criterio 1), y se usa como metrica binaria de precision en el analisis comparativo.

---

#### Limitaciones del enfoque

1. **Sin ancla de verdad absoluta.** Al no haber gold answer, no es posible determinar si el agente y el juez cometen el mismo error razonando en la misma direccion. Si ambos modelos comparten un sesgo sistematico sobre una clausula especifica, la evaluacion no lo detectara.

2. **Dependencia del modelo juez.** La calidad de las puntuaciones esta acotada por la capacidad del juez para razonar sobre ISO 27001. Se utiliza `claude-sonnet-4` con temperatura 0 para maximizar consistencia, pero el juez no es infalible.

3. **Varianza en casos ambiguos.** En clausulas con evidencia parcial o contradictoria, el juez puede asignar puntuaciones distintas a respuestas igualmente validas dependiendo del orden en que procesa la informacion.

4. **Metrica de comparacion relativa, no absoluta.** Una puntuacion de 80/100 no certifica que el hallazgo sea correcto; certifica que es coherente con la evidencia disponible. El benchmark sirve para ordenar modelos, no para validar hallazgos individuales.


In [44]:
# Select 10 clauses at random for a prompt cases
gob_ref_ids = random.sample(config['agents']['gobernanza']['ref_ids'], 10)
for ref_id in gob_ref_ids:
    text = u.build_clause_text(ISO, ref_id)
    print(text)
    print('\n-------')

Clausula 9.3 - Revisión por la dirección
La alta dirección debe revisar el sistema de gestión de la seguridad de la información de la organización a intervalos planificados.
Requisitos especificos:
a) Ejecutar revisiones a intervalos planificados por parte de la alta dirección.
b) Incluir estados de acciones de revisiones previas y cambios en cuestiones externas/internas del apartado 4.1.
c) Revisar la retroalimentación sobre el desempeño de seguridad (tendencias de no conformidades, monitoreo, auditorías y objetivos).
d) Revisar la retroalimentación de las partes interesadas (4.2), resultados de evaluaciones de riesgo y estado de planes de tratamiento.
e) Evaluar oportunidades de mejora continua y la necesidad de cambios en el SGSI.
f) Incluir decisiones y salidas sobre oportunidades de mejora continua y retener evidencia documentada.

-------
Clausula 9.2 - Auditoría interna
La organización debe llevar a cabo auditorías internas a intervalos planificados para proporcionar información

In [55]:
# Import the prompt
gov_prompt = u.load_prompt('./prompts/governance_agent.txt')

In [79]:
# Import the governance evaluation dataset/cases
with open('./evaluation-sets/gov_eval_set.json', 'r', encoding='utf8') as f:
    records = json.load(f)

In [81]:
# This an example of the agent
record_prompt = u.build_agent_message(records[1], gov_prompt)
print(record_prompt[-1800:])

nt_Report_Mar2025.pdf",
    "Incident_Report_Jun2025.pdf"
  ]
}
</expected_output>
</example>

---
<retrieved_context>
[
  {
    "document": "Manual_SGSI_v2.pdf",
    "content": "Sección 9.2: La organización realizará auditorías internas periódicamente para verificar el cumplimiento del SGSI."
  },
  {
    "document": "Plan_Operativo_Anual_2025.pdf",
    "content": "Cronograma de actividades 2025: Se programó una revisión interna de archivos en el mes de Septiembre a cargo del Administrador del SGSI."
  }
]
</retrieved_context>

<clause_to_evaluate>
Clausula 9.2 - Auditoría interna
La organización debe llevar a cabo auditorías internas a intervalos planificados para proporcionar información sobre el estado del SGSI.
Requisitos especificos:
a) Se conforma con los requisitos propios de la organización para su sistema de gestión de la seguridad de la información.
b) Se conforma con los requisitos de esta Norma Internacional.
c) Se implementa y se mantiene eficazmente.
d) Planificar, estab